In [122]:
# se importan las principales librerias 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [123]:
df = pd.read_csv("../data/customer_acquisition_data.csv")
df.head()

,customer_id,channel,cost,conversion_rate,revenue
0,1,referral,8.320327,0.123145,4199
1,2,paid advertising,30.450327,0.016341,3410
2,3,email marketing,5.246263,0.043822,3164
3,4,social media,9.546326,0.167592,1520
4,5,referral,8.320327,0.123145,2419


## Data Cleaning y analisis descriptivo
Para esta seccion vamos a :
- Revisar calidad de los datos
- analizar nulos y tipos
- hacer analisis descriptivo
- analizar canales 
- analizar revenue, cost y conversion
- feature engineering : net_value
- evaluar eficiencia por canal

In [124]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      800 non-null    int64  
 1   channel          800 non-null    str    
 2   cost             800 non-null    float64
 3   conversion_rate  800 non-null    float64
 4   revenue          800 non-null    int64  
dtypes: float64(2), int64(2), str(1)
memory usage: 41.3 KB


In [125]:
# estadisiticas descriptivas. 
df.describe()

,customer_id,cost,conversion_rate,revenue
count,800.0000,800.000000,800.000000,800.000000
mean,400.5000,13.148052,0.086305,2769.151250
std,231.0844,9.922337,0.059611,1259.543706
min,1.0000,5.246263,0.016341,500.000000
25%,200.7500,5.246263,0.043822,1694.000000
50%,400.5000,8.320327,0.043822,2764.000000
75%,600.2500,9.546326,0.123145,3824.250000
max,800.0000,30.450327,0.167592,4998.000000


Las estadisticias nos muestran que: 

1. Las observaciones presentan un revenue promedio de aproximadamente USD 2,769.

2. La tasa de conversión promedio es de 8.63%. Esto significa que, en promedio, se producen aproximadamente 8.63 conversiones por cada 100 oportunidades

3. El 75% de las observaciones presenta un revenue igual o inferior a USD 3,824.25, mientras que el 25% restante supera este valor.

4.  Los valores de revenue presentan una dispersión considerable alrededor de la media, con una desviación estándar de aproximadamente USD 1,260

In [126]:
df["channel"].value_counts()

channel
email marketing     214
referral            207
paid advertising    194
social media        185
Name: count, dtype: int64

In [127]:
df["channel"].value_counts(normalize=True)*100

channel
email marketing     26.750
referral            25.875
paid advertising    24.250
social media        23.125
Name: proportion, dtype: float64

El dataset esta relativamente balanceado en los diferentes canalaes. email marketin representa (26.75), seguido por referral (25.8), paid adevertising (24.25) y social media (23.12)  social media (23.13%)

In [128]:
# Generacion de revenue por canal
revenue_by_channel = df.groupby('channel')['revenue'].sum().reset_index()
revenue_by_channel.sort_values(by='revenue', ascending=False, inplace=True)
revenue_by_channel 

,channel,revenue
0,email marketing,604706
2,referral,569552
1,paid advertising,548396
3,social media,492667


El canal que mas ingresos registro por canal fue email marketing, seguido de paid referral, paid advertising y social media. 

In [129]:
average_revenue_by_channel = df.groupby('channel')['revenue'].mean().reset_index()
average_revenue_by_channel.sort_values(by='revenue', ascending=False, inplace=True)
average_revenue_by_channel

,channel,revenue
1,paid advertising,2826.783505
0,email marketing,2825.728972
2,referral,2751.458937
3,social media,2663.064865


Al analizar el revenue promedio por canal, observamos que Paid Advertising registra el mayor ingreso promedio por observación (USD 2,826.78), seguido de Email Marketing (USD 2,825.73), Referral (USD 2,751.45) y Social Media (USD 2,663.06).

Es importante destacar que Paid Advertising representa aproximadamente el 24% de las observaciones; sin embargo, registra el mayor revenue promedio entre los canales analizados



In [130]:
# Tasa de conversion por canal
conversion_rate_by_channel = df.groupby('channel')['conversion_rate'].mean().reset_index()
conversion_rate_by_channel.sort_values(by='conversion_rate', ascending=False, inplace=True)
conversion_rate_by_channel 

,channel,conversion_rate
3,social media,0.167592
2,referral,0.123145
0,email marketing,0.043822
1,paid advertising,0.016341


El canal que presenta una mayor tasa de conversion es social media (16,75%), seguido por referral(12,31%), email marketing(4,38%) y paid advertising (1,63%)

In [131]:
# usamos merge para unir los dataframes de revenue y conversion rate
channel_metrics = pd.merge(average_revenue_by_channel, conversion_rate_by_channel, on='channel')
channel_metrics.sort_values(by='revenue', ascending=False, inplace=True)
channel_metrics

,channel,revenue,conversion_rate
0,paid advertising,2826.783505,0.016341
1,email marketing,2825.728972,0.043822
2,referral,2751.458937,0.123145
3,social media,2663.064865,0.167592


Aunque Paid Advertising representa el mayor costo entre los canales y presenta la menor tasa de conversión (1.63%), registra el mayor revenue promedio (USD 2,826.78). Esto evidencia que una mayor tasa de conversión no necesariamente se traduce en un mayor revenue promedio.

El bajo nivel de conversión de Paid Advertising representa una oportunidad potencial de optimización: **si la empresa logra incrementar su tasa de conversión manteniendo el valor promedio de los ingresos, podría aumentar significativamente el desempeño del canal**.

Por otro lado, Email Marketing presenta una tasa de conversión relativamente baja (4.38%), pero también debemos considerar su nivel de costos antes de determinar si es un canal eficiente o rentable.

In [132]:
# costo por canal
cost_by_channel = df.groupby('channel')['cost'].mean().reset_index()
cost_by_channel.sort_values(by='cost', ascending=False, inplace=True)
cost_by_channel

,channel,cost
1,paid advertising,30.450327
3,social media,9.546326
2,referral,8.320327
0,email marketing,5.246263


In [133]:
# Aplicamos feature engineering para crear la columna net value y net_value / cost ratio
df["net_value"] = df["revenue"] - df["cost"].round(3)
df["net_value_cost_ratio"] = df["net_value"] / df["cost"].round(3)
df.head()

,customer_id,channel,cost,conversion_rate,revenue,net_value,net_value_cost_ratio
0,1,referral,8.320327,0.123145,4199,4190.680,503.687500
1,2,paid advertising,30.450327,0.016341,3410,3379.550,110.986864
2,3,email marketing,5.246263,0.043822,3164,3158.754,602.126191
3,4,social media,9.546326,0.167592,1520,1510.454,158.228996
4,5,referral,8.320327,0.123145,2419,2410.680,289.745192


In [134]:
# tabla con valor neto por canal
net_value_by_channel = df.groupby('channel')['net_value'].mean().reset_index()
net_value_by_channel.sort_values(by='net_value', ascending=False, inplace=True)
net_value_by_channel

,channel,net_value
0,email marketing,2820.482972
1,paid advertising,2796.333505
2,referral,2743.138937
3,social media,2653.518865


In [135]:
# tabla del costo promedio por ratio de valor neto por canal
net_value_cost_ratio_by_channel = df.groupby('channel')['net_value_cost_ratio'].mean().reset_index()
net_value_cost_ratio_by_channel.sort_values(by='net_value_cost_ratio', ascending=False, inplace=True)
net_value_cost_ratio_by_channel

,channel,net_value_cost_ratio
0,email marketing,537.644486
2,referral,329.704199
3,social media,277.971807
1,paid advertising,91.833613


In [136]:
# unimos los dataframes de average_revenue_by_channel, cost_by_channel, net_value_by_channel y net_value_cost_ratio_by_channel
revenue_by_channel_metrics = pd.merge(
    average_revenue_by_channel,
    cost_by_channel,
    on="channel"
)

revenue_by_channel_metrics = pd.merge(
    revenue_by_channel_metrics,
    net_value_by_channel,
    on="channel"
)

revenue_by_channel_metrics = pd.merge(
    revenue_by_channel_metrics,
    net_value_cost_ratio_by_channel,
    on="channel"
)

In [137]:
revenue_by_channel_metrics.sort_values(by='net_value_cost_ratio', ascending=False, inplace=True) 
revenue_by_channel_metrics

,channel,revenue,cost,net_value,net_value_cost_ratio
1,email marketing,2825.728972,5.246263,2820.482972,537.644486
2,referral,2751.458937,8.320327,2743.138937,329.704199
3,social media,2663.064865,9.546326,2653.518865,277.971807
0,paid advertising,2826.783505,30.450327,2796.333505,91.833613


El canal Paid Advertising genera el mayor revenue promedio; sin embargo, es el canal menos eficiente en términos de valor generado respecto al costo. Por el contrario, Email Marketing genera prácticamente el mismo revenue promedio que Paid Advertising, pero con un costo significativamente menor, lo que se traduce en un net value-to-cost ratio considerablemente superior